# NB_FABRIC_PURVIEW_LINEAGE_TABLE_COLUMN_EXTRACTOR
Extracts table and column-level lineage from Microsoft Fabric Lakehouses and registers it in Microsoft Purview. Supports creating lineage processes with column mapping for data governance and asset tracking.

## Prerequisites
Before running this notebook, ensure the following are in place:

### Service Principal
A dedicated **service principal (App Registration)** is required to authenticate against Microsoft Purview.

- 1 | Create a service principal in Azure Active Directory (App Registration) |
- 2 | Store the `tenant_id`, `client_id`, and `client_secret` in **Azure Key Vault** using the secret names configured in the parameters cell |

### Microsoft Fabric — Workspace Access
The service principal must be added as a **Viewer** to each data workspace that contains the source and target Lakehouses.

> Navigate to each Fabric workspace → **Manage access** → Add the service principal as **Viewer**.

### Microsoft Purview — Role Assignment
The service principal must have one of the following roles assigned in Microsoft Purview to register lineage and manage data assets:

- **Data Curator** — grants read/write access to data assets and lineage
- **Data Source Admin** — grants full control over data source registration and scanning

> Navigate to **Microsoft Purview** → **Data Map** → **Collections** → select your collection → **Role assignments** → add the service principal to the desired role.

## Parameters

- tenant_id="**tenantid**"                              #or your own secret name
- client_id="**sp-fabric-purview-deployment-appid**"    #or your own secret name
- key_vault ="**key_vault_uri_name**"  
- secret_name="**sp-fabric-purview-deployment-secret**" #or your own secret name
- PurviewAccount_name="**Purview_account**"
- process_type="**ShortCut, Notebook, MLV**" what kind of lineage are you creating

## Remarks
- Tablenames must be the same in source and target(sink)
- Tableschema is optional but more reliable
- Columns without mapping will be marked in Purview with a * 
- The Lineage extractor can only be used when extracting tables(files are not supported) 
- All Lineage items can be found in the **Microsoft Purview** → **Unified Catalog** → **Data assets** → **Browse by source type** → **Search results** → **Fabric to Purview Lineage Extractor Process**


## Limitations
- Only one source and one target can be defined (one to one relation)
- Only Lakehouses are supported


In [89]:
pip install pyapacheatlas

StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 91, Finished, Available, Finished, False)

Note: you may need to restart the kernel to use updated packages.


In [90]:
import requests
import os
from pyspark.sql.types import StructType,StructField, StringType, IntegerType
from pyspark.sql.functions import col, when, lower
import json
import pyodbc
import struct
from azure.identity import ClientSecretCredential
# pyapacheatlas

from pyapacheatlas.auth import ServicePrincipalAuthentication
from pyapacheatlas.core import PurviewClient, AtlasEntity, AtlasProcess
from pyapacheatlas.core.typedef import AtlasAttributeDef, EntityTypeDef
from pyapacheatlas.core.util import AtlasException, GuidTracker
from pyapacheatlas.core import EntityTypeDef, RelationshipTypeDef

StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 92, Finished, Available, Finished, False)

In [91]:
SourceWorkspaceName='INTEGRATION DATA (D)'
TargetWorkspaceName='SALES DATA (D)'
SourceLakehouseName='LH_SILVER_LAYER'
TargetLakehouseName='LH_SILVER_LAYER'


filter='Purchasing_PurchaseOrders'       #add filter if you only want to test with one table

StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 93, Finished, Available, Finished, False)

In [92]:
config = {
    "source": {
        "workspace_name": SourceWorkspaceName,
        "lakehouse_name": SourceLakehouseName

    },
    "target": {
        "workspace_name": TargetWorkspaceName,
        "lakehouse_name": TargetLakehouseName
    }
}

StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 94, Finished, Available, Finished, False)

In [93]:
# --------------------------------------------------------------------------------------
# Configuration / Parameters
# Supply via environment variables or by directly editing defaults below.
# --------------------------------------------------------------------------------------

tenant_id="tenantid"
client_id="sp-fabric-purview-deployment-appid"
key_vault ="ncceuwvaultoxgn01"
secret_name="sp-fabric-purview-deployment-secret"
PurviewAccount_name="ededeunpview01"

InputSource01 ='Silver'  # will be used to define label, you can change this based on you requirements
OutputSource01 = 'Silver' # will be used to define label, you can change this based on you requirements

processtype='Shortcut'   #Notebook, Shortcut, MLV
fabric_table_type_name = 'fabric_lakehouse_table'   #In case of a Lakehouse leave as is


driver = '{ODBC Driver 18 for SQL Server}'
process_type_name = "Fabric to Purview Lineage_Extractor_Process"
sourceschema_is_targetschema=False    #If schema in source is differenten from targetschema set this to False, check will be done on TableName. If set to True check will be done on TableSchema, TableName(more realible)


StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 95, Finished, Available, Finished, False)

### Get secrets from Keyvault

In [94]:
notebook_key_vault = f'https://{key_vault}.vault.azure.net/'
purview_tenant_id = notebookutils.credentials.getSecret(notebook_key_vault, tenant_id) 
purview_client_id  = notebookutils.credentials.getSecret(notebook_key_vault, client_id) 
purview__client_secret  = notebookutils.credentials.getSecret(notebook_key_vault, secret_name) 

StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 96, Finished, Available, Finished, False)

In [95]:
def get_or_create_entity(
    entity_name: str,
    type_name: str,
    qualified_name: str,
    temp_guid: str = "-100"
):
    """
    Returns an AtlasEntity.
    If the entity already exists, the existing GUID is reused.
    Otherwise a new AtlasEntity is returned with a temporary negative GUID.
    """

    try:
        existing = client.get_entity(
            qualifiedName=qualified_name,
            typeName=type_name,
        )

        entity_guid = existing["entities"][0]["guid"]

        print("✅ Existing entity found")
        print(f"   Name: {existing['entities'][0]['attributes']['name']}")
        print(f"   QualifiedName: {qualified_name}")
        print(f"   GUID: {entity_guid}")

        return AtlasEntity(
            name=entity_name,
            typeName=type_name,
            qualified_name=qualified_name,
            guid=entity_guid
        )

    except Exception:
        print("🆕 No existing entity found → creating new one")
        print(f"   QualifiedName: {qualified_name}")

        return AtlasEntity(
            name=entity_name,
            typeName=type_name,
            qualified_name=qualified_name,
            guid=temp_guid
        )

def create_lineage_process(
    input_entity: str,
    output_entity: str,
    process_type_name: str,
    process_name: str,
    process_qn: str,
    temp_guid: str = "-100",
    labels=None,
    column_mapping=None
):
    """
    Create or update a Purview lineage process.
    """
    try:
        try:
            existing = client.get_entity(
                qualifiedName=process_qn,
                typeName=process_type_name
            )
            process_guid = existing["entities"][0]["guid"]
        except:
            process_guid = "-1"  # new process
        labels = labels or []
        process_guid = temp_guid if process_guid == "-1" else process_guid
        process = AtlasProcess(
        name=process_name,
        typeName=process_type_name,
        description="Lineage Created by Fabric Purview Accelerator",
        qualified_name=process_qn,
        labels=[labels],
        inputs=[input_entity],
        outputs=[output_entity],
        guid=process_guid,
        attributes={
            "columnMapping": json.dumps(column_mapping),

            "userDescription": f'<div>Lineage<p>Created by Fabric Purview Accelerator</p> <p> Process {process_qn}</p>'

        },
        # This custom attribute flips a switch inside of the Purview UI to render
        # the rich text description.
        customAttributes={
            "microsoft_isDescriptionRichText": "true"
        }
        )
        results = client.upload_entities(
                batch=[
                    input_entity,
                    output_entity,
                    process
                ]
            )
        print(f"   QualifiedName: {process_qn}")
        print("✅ Lineage created")
        process_guid = results["guidAssignments"].get(str(process.guid), process.guid)
        print(f'Search for "{process.name}" or use guid {process_guid}')

    except Exception as e:
        print(f"ERROR: {type(e).__name__}: {e}")
        return None, process_guid


StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 97, Finished, Available, Finished, False)

In [96]:
sqlquery=(
f"SELECT "
    f"s.name  AS SchemaName, "
    f"t.name  AS TableName, "
    f"c.name  AS ColumnName, "
    f"c.column_id as ColumnId, "
    f"ty.name AS ColumnType "
f"FROM sys.tables t "
f"INNER JOIN sys.schemas s ON t.schema_id = s.schema_id "
f"INNER JOIN sys.columns c ON t.object_id = c.object_id "
f"INNER JOIN sys.types AS ty ON c.user_type_id=ty.user_type_id"
f" where 1=1"
)
sqlquery_whereclause=(f" and t.name ='{filter}'" )


StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 98, Finished, Available, Finished, False)

In [97]:
if filter!='':
    sqlquery=sqlquery+sqlquery_whereclause

StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 99, Finished, Available, Finished, False)

In [98]:
#table to store the metadata
df_columns_row={"WorkspaceName": "dummy_column1", "WorkspaceID": "dummy_id", \
        "LakehouseName": "", \
        "LakehouseID": "", \
        "TableName": "", \
        "TableSchema": "", \
        "ColumnName": "", \
        "ColumnType": "", \
        "ColumnID": "", \
        "PurviewFQN":""}

schema_df_columns = StructType([ \
    StructField("WorkspaceName",StringType(),True), \
    StructField("WorkspaceID",StringType(),True), \
    StructField("LakehouseName",StringType(),True), \
    StructField("LakehouseID",StringType(),True), \
    StructField("TableName",StringType(),True), \
    StructField("TableSchema",StringType(),True), \
    StructField("ColumnName", StringType(), True), \
    StructField("ColumnType", StringType(), True), \
    StructField("ColumnID", StringType(), True), \
    StructField("PurviewFQN", StringType(), True) \
  ])
 
df_columns = spark.createDataFrame(data=[],schema=schema_df_columns)


StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 100, Finished, Available, Finished, False)

In [99]:
#Authentication to Fabric workspace. 

sqlclient = ClientSecretCredential(
        tenant_id=purview_tenant_id,
        client_id=purview_client_id,
        client_secret=purview__client_secret)


StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 101, Finished, Available, Finished, False)

In [100]:
#Authentication to Purview. Service principal must have data curator or datasource admin role

oauth = ServicePrincipalAuthentication(
        tenant_id=purview_tenant_id,
        client_id=purview_client_id,
        client_secret=purview__client_secret
    )
client = PurviewClient(
        account_name=PurviewAccount_name,
        authentication=oauth
    )


StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 102, Finished, Available, Finished, False)

In [101]:
process_type = EntityTypeDef(
                        name =  process_type_name,
                        serviceType =  process_type_name,
                        
                        attributeDefs = [
                          AtlasAttributeDef(name = "schedule",
                                              defaultValue = "adHoc"),
                          AtlasAttributeDef(name = "createTime"),
                          AtlasAttributeDef(name="dataLayer", isOptional=True),
                          AtlasAttributeDef("columnMapping")
                        ],
                        superTypes = ["Process"]
                    )

# Upload the new asset type definition
typedef_results = client.upload_typedefs(entityDefs=[process_type], force_update=True)
#print(typedef_results)

StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 103, Finished, Available, Finished, False)

In [102]:
# Get token for Fabric authentication
fabric_token = notebookutils.credentials.getToken('https://analysis.windows.net/powerbi/api')

# Get token for Azure SQL authentication
token = notebookutils.credentials.getToken('https://analysis.windows.net/powerbi/api').encode("UTF-16-LE")

access_token = sqlclient.get_token("https://database.windows.net/.default").token

# Azure SQL ODBC expects the token as UTF-16-LE bytes
token_bytes = access_token.encode("UTF-16-LE")

token_struct = struct.pack(
    f"<I{len(token_bytes)}s",
    len(token_bytes),
    token_bytes
)

StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 104, Finished, Available, Finished, False)

In [103]:
def create_fabric_session(fabric_token: str):
    fabric_headers = {
        'Content-Type': 'application/json',
        'Authorization': f'Bearer {fabric_token}'
    }
    fabric_session = requests.Session()
    fabric_session.headers.update(fabric_headers)
    return fabric_session

StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 105, Finished, Available, Finished, False)

In [104]:
fabric_session = create_fabric_session(fabric_token)

StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 106, Finished, Available, Finished, False)

In [105]:
#Read from sys tables from source and target Location, to get Table, Schema and Column

rows = []

workspace_response = fabric_session.get(f"https://api.fabric.microsoft.com/v1/workspaces/")
workspace_response.raise_for_status()
workspace_list = workspace_response.json().get("value", [])
for workspace in workspace_list:
    for environment, settings in config.items():

        workspace_name = settings["workspace_name"]
        if workspace["displayName"] == workspace_name:
            workspace_id = workspace["id"]
            workspace_name = workspace["displayName"]
            expected_lakehouse_name = settings["lakehouse_name"]

            items_response = fabric_session.get(f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/items")
            items_response.raise_for_status()

            artifacts_list = items_response.json().get("value", [])

            for item in artifacts_list:
                if item["type"] == "Lakehouse" and item["displayName"] == expected_lakehouse_name:
                    item_name = item["displayName"]
                    item_type = item["type"]
                    item_id = item["id"]

                    print(f'\n{item_name} in {workspace_name} is of type {item_type} ' f'and has guid {item_id}'  )

                    lakehouse_response = fabric_session.get(
                        f"https://api.fabric.microsoft.com/v1/workspaces/{workspace_id}/lakehouses/{item_id}"
                    )
                    lakehouse_response.raise_for_status()

                    lakehouse_json = lakehouse_response.json()

                    connstring = lakehouse_json["properties"]["sqlEndpointProperties"]["connectionString"]

                    print(f"SQLconn_string for {workspace_name} and {item_name} is {connstring}")

                    conn = None

                    try:
                        conn = pyodbc.connect(
                            f"DRIVER={driver};"
                            f"SERVER={connstring};"
                            f"PORT=1433;"
                            f"DATABASE={item_name};",
                            attrs_before={1256: token_struct},
                            timeout=12
                        )

                        conn.timeout = 10
                        #make sure we get the correct tables from the correct lakehouse
        
                        with conn.cursor() as cursor:
                            # Warm-up
                            cursor.execute("SELECT 1")
                            cursor.fetchone()

                            cursor.execute(sqlquery)
                            column_list = cursor.fetchall()

                            for column in column_list:
                                new_dict = df_columns_row.copy()

                                new_dict.update({
                                    "WorkspaceName": workspace_name,
                                    "WorkspaceID": workspace_id,
                                    "LakehouseName": item_name,
                                    "LakehouseID": item_id,
                                    "TableSchema": column[0],
                                    "TableName": column[1],
                                    "ColumnName": column[2],
                                    "ColumnID": column[3],
                                    "ColumnType": column[4],
                                    "PurviewFQN": "https://app.fabric.microsoft.com/groups/"+workspace_id+"/lakehouses/"+item_id+"/tables/"+column[0].lower()+"%252F"+column[1].lower()


                                    
                                })

                                rows.append(new_dict)

                    finally:
                        if conn is not None:
                            conn.close()
df_columns = spark.createDataFrame(data=rows, schema=schema_df_columns)


StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 107, Finished, Available, Finished, False)


LH_SILVER_LAYER in SALES DATA (D) is of type Lakehouse and has guid 4964058a-f01f-4398-9426-4d86dc40682b
SQLconn_string for SALES DATA (D) and LH_SILVER_LAYER is nl7yhqnbrscude3yv6mas6bxpq-3vnmjcnjptpu7pqprzlgkkkage.datawarehouse.fabric.microsoft.com

LH_SILVER_LAYER in INTEGRATION DATA (D) is of type Lakehouse and has guid 41c29853-15af-44ef-8160-05958bed1cec
SQLconn_string for INTEGRATION DATA (D) and LH_SILVER_LAYER is nl7yhqnbrscude3yv6mas6bxpq-ik3ugpsbh3ae5jauaow7qpai44.datawarehouse.fabric.microsoft.com


In [106]:
print(f"Found Table columns for:")

#display(df_columns)


StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 108, Finished, Available, Finished, False)

Found Table columns for:


In [107]:
# 1. Filter source and sink only once
source_all = df_columns.filter(
    (col("WorkspaceName") == SourceWorkspaceName) &
    (col("LakehouseName") == SourceLakehouseName)
)

sink_all = df_columns.filter(
    (col("WorkspaceName") == TargetWorkspaceName) &
    (col("LakehouseName") == TargetLakehouseName)
)

# 2. Get unique source tables
source_tables = (
    source_all
    .select("TableSchema", "TableName")
    .distinct()
    .collect()
)

StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 109, Finished, Available, Finished, False)

In [108]:
# 3. Loop over each source table
for table in source_tables:
    InputTableSchema01 = table["TableSchema"]
    InputTableName01 = table["TableName"]

    print(f"Processing table: {InputTableSchema01}.{InputTableName01}")

    # Filter source for current table
    source = source_all.filter(
        (col("TableSchema") == InputTableSchema01) &
        (col("TableName") == InputTableName01)
    )

    # Filter sink for matching table
    if sourceschema_is_targetschema:
        sink = sink_all.filter(
                (col("TableSchema") == InputTableSchema01) &
                (col("TableName") == InputTableName01))
        print('Source and Targetschema are the same')
    else:
        sink = sink_all.filter(
               
            (col("TableName") == InputTableName01)
        )
        print('Source and Targetschema are not the same')

    # 4. Full outer join per table
    full_join = source.alias("b").join(
        sink.alias("s"),
        on=[
           # col("b.TableSchema") == col("s.TableSchema"),  schema can be different
            col("b.TableName") == col("s.TableName"),
            col("b.ColumnName") == col("s.ColumnName")
        ],
        how="full_outer"
    )

    rows = full_join.select(
        col("b.ColumnName").alias("Source"),
        col("b.PurviewFQN").alias("SourcePurviewFQN"),
        col("b.TableSchema").alias("InputTableSchema01"),
        col("b.TableName").alias("InputTableName01"),

        col("s.ColumnName").alias("Sink"),
        col("s.PurviewFQN").alias("TargetPurviewFQN"),
        col("s.TableSchema").alias("OutputTableSchema01"),
        col("s.TableName").alias("OutputTableName01")
    ).collect()

    # Skip if there are no rows
    if not rows:
        print(f"⚠️ No columns found for {InputTableSchema01}.{InputTableName01}")
        continue

    # 5. Get table-level values from first row
    qualified_name_input01 = rows[0]["SourcePurviewFQN"]
    qualified_name_output01 = rows[0]["TargetPurviewFQN"]

    InputTableName01 = rows[0]["InputTableName01"]
    OutputTableName01 = rows[0]["OutputTableName01"]

    InputTableSchema01 = rows[0]["InputTableSchema01"]
    OutputTableSchema01 = rows[0]["OutputTableSchema01"]

    # Skip if source or target FQN is missing
    if not qualified_name_input01 or not qualified_name_output01:
        print(f"⚠️ Missing Source or Target PurviewFQN for {InputTableSchema01}.{InputTableName01}")
        continue

    # 6. Create column mapping per table
    column_mapping = [{
        "ColumnMapping": [
            {
                "Source": row["Source"] if row["Source"] else "*",
                "Sink": row["Sink"] if row["Sink"] else "*"
            }
            for row in rows
        ],
        "DatasetMapping": {
            "Source": qualified_name_input01,
            "Sink": qualified_name_output01
        }
    }]

    #print("Column mapping: {column_mapping}")


    # 7. Get or create source Atlas entity
    input_table= get_or_create_entity(entity_name=InputTableName01, type_name=fabric_table_type_name ,qualified_name = qualified_name_input01, temp_guid = "-3")


    # 8. Get or create target Atlas entity
    output_table=get_or_create_entity(entity_name=OutputTableName01, type_name=fabric_table_type_name ,qualified_name = qualified_name_output01, temp_guid = "-4")
    

# 9. Here you we create our AtlasProcess per table(Lineage Object)
    process_qn = (f"{processtype}://{SourceWorkspaceName}/{InputTableSchema01}/{InputTableName01} to {OutputTableSchema01}/{OutputTableName01}")
    process_name=(f"{processtype} Lineage {InputTableSchema01}.{InputTableName01} to {OutputTableSchema01}.{OutputTableName01}")
    results=create_lineage_process(input_entity=input_table, output_entity = output_table, process_type_name =process_type_name, process_name = process_name, process_qn = process_qn,temp_guid='-999', labels=InputSource01, column_mapping=column_mapping)
    print(json.dumps(results, indent=2))

StatementMeta(, 64ab6d1c-c450-4840-b804-f44ce28284b5, 110, Finished, Available, Finished, False)

Processing table: wwi.Purchasing_PurchaseOrders
Source and Targetschema are not the same
✅ Existing entity found
   Name: Purchasing_PurchaseOrders
   QualifiedName: https://app.fabric.microsoft.com/groups/3e43b742-3e41-4ec0-a414-03adf83c08e7/lakehouses/41c29853-15af-44ef-8160-05958bed1cec/tables/wwi%252Fpurchasing_purchaseorders
   GUID: 79fd11c0-a82d-4cd3-89ca-1df6f6f60000
✅ Existing entity found
   Name: Purchasing_PurchaseOrders
   QualifiedName: https://app.fabric.microsoft.com/groups/89c45add-7ca9-4fdf-be0f-8e5665294031/lakehouses/4964058a-f01f-4398-9426-4d86dc40682b/tables/dbo%252Fpurchasing_purchaseorders
   GUID: 75424b06-ad10-4463-ad1b-71f6f6f60000
   QualifiedName: Shortcut://INTEGRATION DATA (D)/wwi/Purchasing_PurchaseOrders to dbo/Purchasing_PurchaseOrders
✅ Lineage created
Search for "Shortcut Lineage wwi.Purchasing_PurchaseOrders to dbo.Purchasing_PurchaseOrders" or use guid e976aa4b-fb8b-43b7-95c5-687d16334156
null
